# Marshmallow — Advanced Problems with Solutions

**Target:** Marshmallow 4.x (current 4.3.x API)

This notebook modernizes older Marshmallow 2.x-style material and turns it into advanced, production-oriented practice.

## You will practice

- `dump`, `dumps`, `load`, `loads`, and `validate`
- `ValidationError` and structured error messages
- `required`, `allow_none`, validators, and schema-level invariants
- strict unknown-field policies: `RAISE`, `EXCLUDE`, `INCLUDE`
- `load_only`, `dump_only`, `data_key`, `only`, `exclude`
- nested schemas, `many=True`, recursive schemas
- `@pre_load`, `@post_load`, `@post_dump`, `pass_collection=True`
- immutable dataclasses as domain models
- correct PATCH/update semantics
- `Decimal` for money
- custom `Field` implementations
- indexed bulk errors and custom error translation
- a full e-commerce order capstone

> **Migration warning:** modern Marshmallow does **not** return `MarshalResult` / `UnmarshalResult` objects with `.data` and `.errors`. `load()` returns the deserialized value and raises `ValidationError` on invalid input.


## Setup

Run this once if needed. The notebook pins the major version to Marshmallow 4 while allowing compatible 4.x updates.


In [ ]:
%pip install -q "marshmallow>=4.3,<5"


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field, replace
from datetime import date, datetime, timezone
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
from pprint import pprint
from typing import Any
from uuid import UUID, uuid4

from marshmallow import (
    EXCLUDE,
    INCLUDE,
    RAISE,
    Schema,
    ValidationError,
    fields,
    post_dump,
    post_load,
    pre_load,
    validate,
    validates,
    validates_schema,
)


## Modern API refresher

- `schema.dump(obj)` → serialized Python-native data
- `schema.dumps(obj)` → JSON string
- `schema.load(mapping)` → validated/deserialized data
- `schema.loads(json_string)` → validated/deserialized data from JSON
- `schema.validate(mapping)` → error dictionary without raising

Validation is an **input/deserialization concern**. `dump()` should not be used as a substitute for validating untrusted input.


In [ ]:
class BasicPersonSchema(Schema):
    first_name = fields.Str(required=True)
    last_name = fields.Str(required=True)
    dob = fields.Date(required=True)


basic = BasicPersonSchema()
raw = {"first_name": "John", "last_name": "Cleese", "dob": "1939-10-27"}

loaded = basic.load(raw)
dumped = basic.dump(loaded)

print("loaded:")
pprint(loaded)
print("dumped:")
pprint(dumped)


In [ ]:
try:
    basic.load({"first_name": "John", "last_name": "Cleese", "dob": "bad-date"})
except ValidationError as exc:
    pprint(exc.messages)


In [ ]:
def try_load(schema: Schema, payload: Any, **kwargs) -> dict[str, Any]:
    try:
        return {"ok": True, "value": schema.load(payload, **kwargs)}
    except ValidationError as exc:
        return {
            "ok": False,
            "errors": exc.messages,
            "valid_data": exc.valid_data,
        }


# Problem 1 — Production-grade `PersonSchema`

## Requirements

Create a schema that:

1. requires first name, last name, email, and date of birth;
2. strips whitespace before validation;
3. lowercases email;
4. rejects blank names;
5. validates email;
6. rejects future DOBs and implausible ages over 130;
7. rejects unknown keys;
8. loads into an immutable `Person` dataclass.


In [ ]:
# Starter idea:
# - @dataclass(frozen=True)
# - validate.Length
# - fields.Email
# - @pre_load
# - @validates("dob")
# - @post_load
# - class Meta: unknown = RAISE


## Solution


In [ ]:
@dataclass(frozen=True)
class Person:
    first_name: str
    last_name: str
    email: str
    dob: date


class PersonSchema(Schema):
    first_name = fields.Str(required=True, validate=validate.Length(min=1, max=80))
    last_name = fields.Str(required=True, validate=validate.Length(min=1, max=80))
    email = fields.Email(required=True)
    dob = fields.Date(required=True)

    class Meta:
        unknown = RAISE

    @pre_load
    def normalize(self, data, **kwargs):
        cleaned = dict(data)  # do not mutate caller-owned input
        for key in ("first_name", "last_name", "email"):
            if isinstance(cleaned.get(key), str):
                cleaned[key] = cleaned[key].strip()
        if isinstance(cleaned.get("email"), str):
            cleaned["email"] = cleaned["email"].lower()
        return cleaned

    @validates("dob")
    def validate_dob(self, value: date, **kwargs):
        today = date.today()
        if value > today:
            raise ValidationError("Date of birth cannot be in the future.")
        if (today - value).days > 130 * 365.25:
            raise ValidationError("Date of birth is implausibly old.")

    @post_load
    def make_person(self, data, **kwargs):
        return Person(**data)


person_schema = PersonSchema()


In [ ]:
person = person_schema.load({
    "first_name": "  Alan  ",
    "last_name": " Turing ",
    "email": " ALAN@EXAMPLE.COM ",
    "dob": "1912-06-23",
})

print(person)
pprint(person_schema.dump(person))
assert person.email == "alan@example.com"


In [ ]:
invalid_people = [
    {"first_name": "", "last_name": "Turing", "email": "a@example.com", "dob": "1912-06-23"},
    {"first_name": "Alan", "last_name": "Turing", "email": "bad", "dob": "1912-06-23"},
    {"first_name": "Alan", "last_name": "Turing", "email": "a@example.com", "dob": "2999-01-01"},
    {"first_name": "Alan", "last_name": "Turing", "email": "a@example.com", "dob": "1912-06-23", "admin": True},
]

for n, payload in enumerate(invalid_people, 1):
    print(f"case {n}")
    pprint(try_load(person_schema, payload))


# Problem 2 — Unknown fields: `RAISE`, `EXCLUDE`, `INCLUDE`

Demonstrate all three policies with the same payload and identify when each is useful.


## Solution


In [ ]:
class DeveloperSchema(Schema):
    name = fields.Str(required=True)
    language = fields.Str(required=True)


payload = {"name": "Grace", "language": "COBOL", "unexpected": 123}

for policy in (RAISE, EXCLUDE, INCLUDE):
    schema = DeveloperSchema(unknown=policy)
    print("policy:", policy)
    pprint(try_load(schema, payload))


**Guidance:** use `RAISE` for strict write/API boundaries, `EXCLUDE` when intentionally ignoring upstream additions, and `INCLUDE` only when unmodeled fields must remain available.


# Problem 3 — `load_only`, `dump_only`, and `data_key`

Build a user API contract that:

- accepts `displayName` externally but uses `display_name` internally;
- accepts a password but never dumps it;
- dumps server-generated `id` and `createdAt`;
- validates a minimum 12-character password;
- rejects unknown keys.


## Solution


In [ ]:
class UserContractSchema(Schema):
    user_id = fields.UUID(dump_only=True, data_key="id")
    display_name = fields.Str(required=True, data_key="displayName", validate=validate.Length(min=2, max=80))
    email = fields.Email(required=True)
    password = fields.Str(required=True, load_only=True, validate=validate.Length(min=12, max=256))
    created_at = fields.AwareDateTime(dump_only=True, data_key="createdAt")

    class Meta:
        unknown = RAISE


user_contract = UserContractSchema()


In [ ]:
registration = user_contract.load({
    "displayName": "Margaret Hamilton",
    "email": "margaret@example.com",
    "password": "correct-horse-battery-staple",
})
pprint(registration)


In [ ]:
server_user = {
    "user_id": uuid4(),
    "display_name": "Margaret Hamilton",
    "email": "margaret@example.com",
    "password": "MUST_NOT_LEAK",
    "created_at": datetime.now(timezone.utc),
}

response = user_contract.dump(server_user)
pprint(response)
assert "password" not in response


In [ ]:
pprint(try_load(user_contract, {
    "id": str(uuid4()),
    "displayName": "Edsger Dijkstra",
    "email": "edsger@example.com",
    "password": "a-very-long-password",
    "createdAt": "2026-01-01T12:00:00+00:00",
}))


# Problem 4 — Nested objects and collection invariants

Model a movie with a non-empty actor list. Actor emails must be unique within the movie, and nested dictionaries should load into dataclasses.


## Solution


In [ ]:
@dataclass(frozen=True)
class Actor:
    name: str
    email: str


@dataclass(frozen=True)
class Movie:
    title: str
    year: int
    actors: tuple[Actor, ...]


class ActorSchema(Schema):
    name = fields.Str(required=True, validate=validate.Length(min=1, max=100))
    email = fields.Email(required=True)

    class Meta:
        unknown = RAISE

    @post_load
    def make_actor(self, data, **kwargs):
        return Actor(**data)


class MovieSchema(Schema):
    title = fields.Str(required=True, validate=validate.Length(min=1, max=200))
    year = fields.Int(required=True, strict=True, validate=validate.Range(min=1888, max=date.today().year + 5))
    actors = fields.List(fields.Nested(ActorSchema), required=True, validate=validate.Length(min=1, max=100))

    class Meta:
        unknown = RAISE

    @validates_schema
    def unique_actor_emails(self, data, **kwargs):
        actors = data.get("actors", [])
        emails = [a.email.casefold() for a in actors]
        if len(emails) != len(set(emails)):
            raise ValidationError({"actors": ["Actor emails must be unique within a movie."]})

    @post_load
    def make_movie(self, data, **kwargs):
        data["actors"] = tuple(data["actors"])
        return Movie(**data)


movie_schema = MovieSchema()


In [ ]:
movie = movie_schema.load({
    "title": "Example Film",
    "year": 2025,
    "actors": [
        {"name": "Actor One", "email": "one@example.com"},
        {"name": "Actor Two", "email": "two@example.com"},
    ],
})

print(movie)
pprint(movie_schema.dump(movie))


In [ ]:
pprint(try_load(movie_schema, {
    "title": "Duplicate Cast",
    "year": 2025,
    "actors": [
        {"name": "A", "email": "same@example.com"},
        {"name": "B", "email": "SAME@example.com"},
    ],
}))


# Problem 5 — Schema-level cross-field validation

Validate a booking where datetimes must be timezone-aware, `end > start`, duration ≤ 30 days, and guest count ≤ room capacity.


## Solution


In [ ]:
class BookingSchema(Schema):
    start = fields.AwareDateTime(required=True)
    end = fields.AwareDateTime(required=True)
    guests = fields.Int(required=True, strict=True, validate=validate.Range(min=1, max=100))
    room_capacity = fields.Int(required=True, strict=True, validate=validate.Range(min=1, max=100))

    class Meta:
        unknown = RAISE

    @validates_schema
    def validate_booking(self, data, **kwargs):
        errors = {}
        start, end = data.get("start"), data.get("end")
        guests, capacity = data.get("guests"), data.get("room_capacity")

        if start is not None and end is not None:
            if end <= start:
                errors.setdefault("end", []).append("End must be after start.")
            elif (end - start).days > 30:
                errors.setdefault("end", []).append("Booking cannot exceed 30 days.")

        if guests is not None and capacity is not None and guests > capacity:
            errors.setdefault("guests", []).append("Guest count cannot exceed room capacity.")

        if errors:
            raise ValidationError(errors)


booking_schema = BookingSchema()


In [ ]:
pprint(try_load(booking_schema, {
    "start": "2026-08-10T12:00:00+03:00",
    "end": "2026-08-09T12:00:00+03:00",
    "guests": 6,
    "room_capacity": 4,
}))

pprint(try_load(booking_schema, {
    "start": "2026-08-10T12:00:00",
    "end": "2026-08-11T12:00:00",
    "guests": 2,
    "room_capacity": 4,
}))


# Problem 6 — Correct PATCH/update design

Use separate create and patch schemas so partial updates do not collide with a `@post_load` constructor that expects a complete object.


## Solution


In [ ]:
@dataclass(frozen=True)
class Profile:
    display_name: str
    email: str
    bio: str


class ProfileFieldsSchema(Schema):
    display_name = fields.Str(validate=validate.Length(min=2, max=80))
    email = fields.Email()
    bio = fields.Str(validate=validate.Length(max=500))

    class Meta:
        unknown = RAISE


class ProfileCreateSchema(ProfileFieldsSchema):
    display_name = fields.Str(required=True, validate=validate.Length(min=2, max=80))
    email = fields.Email(required=True)
    bio = fields.Str(required=True, validate=validate.Length(max=500))

    @post_load
    def make_profile(self, data, **kwargs):
        return Profile(**data)


class ProfilePatchSchema(ProfileFieldsSchema):
    @validates_schema
    def reject_empty_patch(self, data, **kwargs):
        if not data:
            raise ValidationError("At least one field must be supplied.")


create_profile = ProfileCreateSchema()
patch_profile = ProfilePatchSchema()


In [ ]:
profile = create_profile.load({
    "display_name": "Guido",
    "email": "guido@example.com",
    "bio": "Python enthusiast.",
})

patch = patch_profile.load({"display_name": "Guido van Rossum"})
updated = replace(profile, **patch)

print("before:", profile)
print("after: ", updated)


In [ ]:
pprint(try_load(patch_profile, {}))
pprint(try_load(patch_profile, {"email": "not-an-email"}))
pprint(try_load(patch_profile, {"role": "admin"}))


`partial=True` is still useful, but a dedicated patch schema often communicates intent better and avoids incomplete object-construction hooks.


# Problem 7 — Exact money with `Decimal`

Create an invoice-line schema with strict integer quantity, exact decimal price, two-place rounding, JSON-safe string serialization, and a computed line total.


## Solution


In [ ]:
@dataclass(frozen=True)
class InvoiceLine:
    sku: str
    quantity: int
    unit_price: Decimal

    @property
    def total(self) -> Decimal:
        return (self.unit_price * self.quantity).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)


class InvoiceLineSchema(Schema):
    sku = fields.Str(required=True, validate=[
        validate.Length(min=3, max=32),
        validate.Regexp(r"^[A-Z0-9][A-Z0-9_-]*$"),
    ])
    quantity = fields.Int(required=True, strict=True, validate=validate.Range(min=1, max=10_000))
    unit_price = fields.Decimal(
        required=True,
        places=2,
        rounding=ROUND_HALF_UP,
        as_string=True,
        validate=validate.Range(min=Decimal("0.00")),
    )

    class Meta:
        unknown = RAISE

    @post_load
    def make_line(self, data, **kwargs):
        return InvoiceLine(**data)


invoice_line_schema = InvoiceLineSchema()


In [ ]:
line = invoice_line_schema.load({"sku": "BOOK-001", "quantity": 3, "unit_price": "19.995"})
print(line)
print("total:", line.total)
pprint(invoice_line_schema.dump(line))
assert line.unit_price == Decimal("20.00")


In [ ]:
for payload in [
    {"sku": "x", "quantity": 1, "unit_price": "10.00"},
    {"sku": "GOOD-SKU", "quantity": 0, "unit_price": "10.00"},
    {"sku": "GOOD-SKU", "quantity": 1.5, "unit_price": "10.00"},
    {"sku": "GOOD-SKU", "quantity": 1, "unit_price": "-0.01"},
]:
    pprint(try_load(invoice_line_schema, payload))


# Problem 8 — Reusable custom `MoneyField`

External form:

```json
{"currency": "USD", "amount": "19.99"}
```

Internal form:

```python
Money(currency="USD", amount=Decimal("19.99"))
```

Support `USD`, `EUR`, and `GBP`; reject negative/invalid amounts; quantize to two decimals.


## Solution


In [ ]:
@dataclass(frozen=True)
class Money:
    currency: str
    amount: Decimal


class MoneyField(fields.Field):
    default_error_messages = {
        "type": "Not a valid money object.",
        "currency": "Unsupported currency.",
        "amount": "Not a valid non-negative decimal amount.",
    }
    allowed_currencies = {"USD", "EUR", "GBP"}

    def _serialize(self, value, attr, obj, **kwargs):
        if value is None:
            return None
        if not isinstance(value, Money):
            raise ValidationError(self.error_messages["type"])
        return {
            "currency": value.currency,
            "amount": format(value.amount.quantize(Decimal("0.01")), "f"),
        }

    def _deserialize(self, value, attr, data, **kwargs):
        if not isinstance(value, dict):
            raise ValidationError(self.error_messages["type"])

        errors = {}
        currency = value.get("currency")
        if currency not in self.allowed_currencies:
            errors["currency"] = [self.error_messages["currency"]]

        try:
            amount = Decimal(str(value.get("amount"))).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
            if amount < 0:
                raise InvalidOperation
        except (InvalidOperation, ValueError, TypeError):
            amount = None
            errors["amount"] = [self.error_messages["amount"]]

        if errors:
            raise ValidationError(errors)
        return Money(currency=currency, amount=amount)


class ProductPriceSchema(Schema):
    product_code = fields.Str(required=True)
    price = MoneyField(required=True)

    class Meta:
        unknown = RAISE


product_price_schema = ProductPriceSchema()


In [ ]:
product = product_price_schema.load({
    "product_code": "P-100",
    "price": {"currency": "EUR", "amount": "9.999"},
})
pprint(product)
pprint(product_price_schema.dump(product))


In [ ]:
for payload in [
    {"product_code": "P-100", "price": "EUR 9.99"},
    {"product_code": "P-100", "price": {"currency": "BTC", "amount": "9.99"}},
    {"product_code": "P-100", "price": {"currency": "USD", "amount": "-1"}},
]:
    pprint(try_load(product_price_schema, payload))


# Problem 9 — Collection envelopes and `pass_collection=True`

Load bulk input from `{ "results": [...] }` into a plain list and wrap a list back into the envelope on dump. Also support single-object `{ "result": ... }` mode.


## Solution


In [ ]:
class ResultSchema(Schema):
    id = fields.Int(required=True, strict=True)
    name = fields.Str(required=True)

    class Meta:
        unknown = RAISE

    @pre_load(pass_collection=True)
    def unwrap(self, data, many, **kwargs):
        key = "results" if many else "result"
        if not isinstance(data, dict) or key not in data:
            raise ValidationError({"_schema": [f"Expected top-level '{key}' envelope."]})
        return data[key]

    @post_dump(pass_collection=True)
    def wrap(self, data, many, **kwargs):
        return {"results" if many else "result": data}


In [ ]:
bulk_schema = ResultSchema(many=True)
bulk_payload = {"results": [{"id": 1, "name": "alpha"}, {"id": 2, "name": "beta"}]}
loaded_results = bulk_schema.load(bulk_payload)
pprint(loaded_results)
pprint(bulk_schema.dump(loaded_results))

single_schema = ResultSchema()
pprint(single_schema.load({"result": {"id": 7, "name": "single"}}))
pprint(single_schema.dump({"id": 7, "name": "single"}))


# Problem 10 — Bulk validation with indexed errors

Validate many members at once and inspect which row failed. Also inspect `ValidationError.valid_data`.


## Solution


In [ ]:
class MemberSchema(Schema):
    name = fields.Str(required=True, validate=validate.Length(min=2, max=80))
    email = fields.Email(required=True)
    age = fields.Int(required=True, strict=True, validate=validate.Range(min=18, max=120))

    class Meta:
        unknown = RAISE
        index_errors = True


members_schema = MemberSchema(many=True)
rows = [
    {"name": "Alice", "email": "alice@example.com", "age": 32},
    {"name": "B", "email": "bad-email", "age": 17},
    {"name": "Charlie", "email": "charlie@example.com", "age": 44},
    {"name": "Dana", "age": 30},
]

try:
    members_schema.load(rows)
except ValidationError as exc:
    print("indexed errors:")
    pprint(exc.messages)
    print("valid_data:")
    pprint(exc.valid_data)


For imports, whether to commit `valid_data` is a domain/transaction decision: some systems require all-or-nothing, while others accept valid rows and report rejected ones.


# Problem 11 — Reusable strict base schema and custom error translation

Create a `StrictSchema`, then translate Marshmallow errors at an API boundary into a domain-specific exception while preserving the structured error dictionary.


## Solution


In [ ]:
class APIInputError(Exception):
    def __init__(self, errors, *, payload=None):
        super().__init__("Invalid API input")
        self.errors = errors
        self.payload = payload


class StrictSchema(Schema):
    class Meta:
        unknown = RAISE


class OrderCommandSchema(StrictSchema):
    product_id = fields.UUID(required=True)
    quantity = fields.Int(required=True, strict=True, validate=validate.Range(min=1, max=100))

    def handle_error(self, exc, data, **kwargs):
        raise APIInputError(exc.messages, payload=data) from exc


In [ ]:
try:
    OrderCommandSchema().load({"product_id": "bad", "quantity": 0, "debug": True})
except APIInputError as exc:
    print(str(exc))
    pprint(exc.errors)
    pprint(exc.payload)


# Problem 12 — Recursive nested schemas

Model a category tree whose `children` recursively contain more categories. Load into immutable objects and dump back to nested dictionaries.


## Solution


In [ ]:
@dataclass(frozen=True)
class Category:
    name: str
    children: tuple["Category", ...] = field(default_factory=tuple)


class CategorySchema(Schema):
    name = fields.Str(required=True, validate=validate.Length(min=1, max=100))
    children = fields.List(fields.Nested(lambda: CategorySchema()), load_default=list)

    class Meta:
        unknown = RAISE

    @post_load
    def make_category(self, data, **kwargs):
        data["children"] = tuple(data["children"])
        return Category(**data)


category_schema = CategorySchema()


In [ ]:
tree_payload = {
    "name": "Programming",
    "children": [
        {"name": "Python", "children": [{"name": "Web"}, {"name": "Data"}]},
        {"name": "Rust"},
    ],
}

tree = category_schema.load(tree_payload)
pprint(category_schema.dump(tree))


In [ ]:
def walk(node: Category, depth=0):
    print("  " * depth + "- " + node.name)
    for child in node.children:
        walk(child, depth + 1)

walk(tree)


# Problem 13 — Response shaping with `only` and `exclude`

Serialize the same movie as a full object, a compact object, an object without year, and an object containing only title + actor names.


## Solution


In [ ]:
print("full:")
pprint(movie_schema.dump(movie))

print("compact:")
pprint(MovieSchema(only=("title", "year")).dump(movie))

print("without year:")
pprint(MovieSchema(exclude=("year",)).dump(movie))

print("title + actor names:")
pprint(MovieSchema(only=("title", "actors.name")).dump(movie))


# Problem 14 — `validate()` versus `load()`

Compare an errors-only validation pass with full deserialization.


## Solution


In [ ]:
candidate = {
    "first_name": "Alan",
    "last_name": "Turing",
    "email": "invalid",
    "dob": "1912-06-23",
}

pprint(person_schema.validate(candidate))

try:
    person_schema.load(candidate)
except ValidationError as exc:
    pprint(exc.messages)


# Problem 15 — Capstone: e-commerce order validation

## Rules

### Customer
- `customerId`: UUID
- valid email

### Line item
- SKU: uppercase letters/numbers/`_`/`-`, 3–32 chars
- strict integer quantity 1–1000
- exact non-negative decimal `unitPrice`

### Order
- unknown keys rejected
- 1–100 line items
- no duplicate SKU values
- timezone-aware `submittedAt`
- client `declaredTotal` must exactly equal the total of all lines
- loads into dataclasses
- decimal values dump as JSON-safe strings
- exposes computed `calculatedTotal` only on dump


## Solution


In [ ]:
@dataclass(frozen=True)
class Customer:
    customer_id: UUID
    email: str


@dataclass(frozen=True)
class OrderLine:
    sku: str
    quantity: int
    unit_price: Decimal

    @property
    def line_total(self) -> Decimal:
        return (self.unit_price * self.quantity).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)


@dataclass(frozen=True)
class Order:
    customer: Customer
    items: tuple[OrderLine, ...]
    submitted_at: datetime
    declared_total: Decimal

    @property
    def calculated_total(self) -> Decimal:
        return sum((item.line_total for item in self.items), start=Decimal("0.00")).quantize(Decimal("0.01"))


In [ ]:
class CustomerSchema(StrictSchema):
    customer_id = fields.UUID(required=True, data_key="customerId")
    email = fields.Email(required=True)

    @post_load
    def make_customer(self, data, **kwargs):
        return Customer(**data)


class OrderLineSchema(StrictSchema):
    sku = fields.Str(required=True, validate=[
        validate.Length(min=3, max=32),
        validate.Regexp(r"^[A-Z0-9][A-Z0-9_-]*$"),
    ])
    quantity = fields.Int(required=True, strict=True, validate=validate.Range(min=1, max=1000))
    unit_price = fields.Decimal(
        required=True,
        data_key="unitPrice",
        places=2,
        rounding=ROUND_HALF_UP,
        as_string=True,
        validate=validate.Range(min=Decimal("0.00")),
    )

    @post_load
    def make_line(self, data, **kwargs):
        return OrderLine(**data)


class OrderSchema(StrictSchema):
    customer = fields.Nested(CustomerSchema, required=True)
    items = fields.List(fields.Nested(OrderLineSchema), required=True, validate=validate.Length(min=1, max=100))
    submitted_at = fields.AwareDateTime(required=True, data_key="submittedAt")
    declared_total = fields.Decimal(
        required=True,
        data_key="declaredTotal",
        places=2,
        rounding=ROUND_HALF_UP,
        as_string=True,
        validate=validate.Range(min=Decimal("0.00")),
    )
    calculated_total = fields.Method(serialize="serialize_total", dump_only=True, data_key="calculatedTotal")

    @validates_schema
    def validate_order(self, data, **kwargs):
        errors = {}
        items = data.get("items", [])
        skus = [item.sku for item in items]

        if len(skus) != len(set(skus)):
            errors.setdefault("items", []).append("Each SKU may appear only once per order.")

        if items and "declared_total" in data:
            expected = sum((item.line_total for item in items), start=Decimal("0.00")).quantize(Decimal("0.01"))
            if data["declared_total"] != expected:
                errors.setdefault("declared_total", []).append(
                    f"Declared total must equal calculated total {expected}."
                )

        if errors:
            raise ValidationError(errors)

    def serialize_total(self, obj):
        return format(obj.calculated_total, ".2f")

    @post_load
    def make_order(self, data, **kwargs):
        data["items"] = tuple(data["items"])
        return Order(**data)


order_schema = OrderSchema()


In [ ]:
valid_order_payload = {
    "customer": {"customerId": str(uuid4()), "email": "buyer@example.com"},
    "items": [
        {"sku": "BOOK-001", "quantity": 2, "unitPrice": "19.95"},
        {"sku": "PEN_002", "quantity": 3, "unitPrice": "1.50"},
    ],
    "submittedAt": "2026-08-07T14:00:00+00:00",
    "declaredTotal": "44.40",
}

order = order_schema.load(valid_order_payload)
print(order)
print("calculated:", order.calculated_total)
pprint(order_schema.dump(order))


In [ ]:
invalid_order_payload = {
    "customer": {"customerId": "not-a-uuid", "email": "not-an-email"},
    "items": [
        {"sku": "BOOK-001", "quantity": 2, "unitPrice": "19.95"},
        {"sku": "BOOK-001", "quantity": 0, "unitPrice": "-1"},
    ],
    "submittedAt": "2026-08-07T14:00:00",
    "declaredTotal": "999.99",
    "debug": True,
}

pprint(try_load(order_schema, invalid_order_payload))


# Extra drills

1. Add a `status` field with `fields.Enum`.
2. Add `fields.IP` and URL validation.
3. Normalize a tag list and reject duplicates after normalization.
4. Validate `OrderSchema(many=True)` and inspect indexed nested errors.
5. Add `currency` to the capstone and reject mixed-currency lines.
6. Add custom error messages at a base-schema level.
7. Accept a legacy input alias while emitting a modern output key.
8. Use Marshmallow 4.3 field-level `pre_load=` / `post_load=` callbacks.
9. Write `pytest` tests for every invalid branch.
10. Benchmark one reused schema instance versus repeatedly constructing schemas in a large loop.


## Extra solution example — normalized unique tags


In [ ]:
class TagSchema(StrictSchema):
    tags = fields.List(
        fields.Str(validate=validate.Length(min=1, max=30)),
        required=True,
        validate=validate.Length(min=1, max=10),
    )

    @pre_load
    def normalize_tags(self, data, **kwargs):
        copied = dict(data)
        if isinstance(copied.get("tags"), list):
            copied["tags"] = [
                item.strip().lower() if isinstance(item, str) else item
                for item in copied["tags"]
            ]
        return copied

    @validates("tags")
    def unique_tags(self, value, **kwargs):
        if len(value) != len(set(value)):
            raise ValidationError("Tags must be unique after normalization.")


tag_schema = TagSchema()
pprint(tag_schema.load({"tags": [" Python ", "API", "serialization"]}))
pprint(try_load(tag_schema, {"tags": ["Python", " python "]}))


## Extra solution example — network fields


In [ ]:
class NetworkIdentitySchema(StrictSchema):
    homepage = fields.Url(required=True)
    ip_address = fields.IP(required=True)


pprint(NetworkIdentitySchema().load({
    "homepage": "https://example.com/profile",
    "ip_address": "2001:db8::1",
}))


## Extra solution example — bulk order errors


In [ ]:
bulk_orders = [
    valid_order_payload,
    {**valid_order_payload, "declaredTotal": "0.01"},
]

try:
    OrderSchema(many=True).load(bulk_orders)
except ValidationError as exc:
    pprint(exc.messages)


# Smoke tests

Move these assertions into `pytest` in a real project. They exercise directionality, normalization, Decimal behavior, PATCH semantics, and recursive round-tripping.


In [ ]:
assert person_schema.load({
    "first_name": "  Alan  ",
    "last_name": " Turing ",
    "email": "ALAN@EXAMPLE.COM ",
    "dob": "1912-06-23",
}).email == "alan@example.com"

assert "password" not in user_contract.dump(server_user)
assert patch_profile.load({"bio": "Updated"}) == {"bio": "Updated"}
assert invoice_line_schema.load({"sku": "ABC-123", "quantity": 2, "unit_price": "1.005"}).unit_price == Decimal("1.01")
assert category_schema.dump(category_schema.load(tree_payload)) == tree_payload
assert order_schema.dump(order)["calculatedTotal"] == "44.40"

print("Smoke assertions passed.")


# Common mistakes to avoid

1. Using old `.data` / `.errors` result objects.
2. Expecting `dump()` to validate untrusted input.
3. Mutating caller-owned data inside `@pre_load`.
4. Using binary floating point for exact money.
5. Accepting unknown keys accidentally.
6. Using a full-object `@post_load` constructor for incomplete PATCH input.
7. Omitting `**kwargs` from decorated methods.
8. Putting simple single-field rules into giant schema-level validators.
9. Treating `only` / `exclude` as authorization.
10. Failing to test nested and indexed error shapes.


# References

- https://marshmallow.readthedocs.io/
- https://marshmallow.readthedocs.io/en/stable/quickstart.html
- https://marshmallow.readthedocs.io/en/stable/marshmallow.schema.html
- https://marshmallow.readthedocs.io/en/stable/marshmallow.fields.html
- https://marshmallow.readthedocs.io/en/stable/marshmallow.validate.html
- https://marshmallow.readthedocs.io/en/stable/custom_fields.html
- https://marshmallow.readthedocs.io/en/stable/upgrading.html

After these exercises, you should be able to treat Marshmallow schemas as explicit, testable boundaries between external/untrusted data and application/domain data.
